<a href="https://www.kaggle.com/code/concacmemay/hybrid-rl-alns-for-vrptw?scriptVersionId=308742086" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

### Application of Reinforcement Learning in Optimizing ALNS for Vehicle Routing with Time Windows

| Algorithm | Description |
|---|---|
| **ALNS** | Baseline — Adaptive Large Neighbourhood Search |
| **DQN** | Ablation — constructive RL (shows why pure RL is insufficient) |
| **RL-ALNS** | 🏆 Proposed — Dueling Double DQN selects operators inside ALNS |
| **RL-ALNS★** | 🏆 Transfer — RL-ALNS pre-trained on RC1, applied zero-shot to RC2 |

**Dataset:** Solomon RC1 + RC2 · 16 instances · 100 customers each · 5 runs

In [1]:
# ── 1. Install & Imports ──────────────────────────────────────────────────────
!pip install numba safetensors scipy -q

import os, glob, time, random, math, copy
from collections import deque
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from numba import njit

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from safetensors.torch import save_file, load_file

import matplotlib
matplotlib.use('Agg')  # headless — no plt.show() blocking
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device : {DEVICE}')
print(f'✅ PyTorch: {torch.__version__}')

✅ Device : cuda
✅ PyTorch: 2.10.0+cu128


In [2]:
# ── 2. Config ─────────────────────────────────────────────────────────────────
IN_KAGGLE  = os.path.exists('/kaggle/working')
DATA_PATH  = ('/kaggle/input/datasets/senju14/vrptw-benchmark-datasets/data/Solomon'
              if IN_KAGGLE else '/content/vrptw-benchmark/data/Solomon')
OUTPUT_DIR = '/kaggle/working' if IN_KAGGLE else '/content'
MODEL_DIR  = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

@dataclass
class Config:
    # ── ALNS ──────────────────────────────────────────────────────────────────
    alns_iterations:     int   = 1_200   # baseline budget
    rla_iterations:      int   = 1_800   # RL-ALNS gets more: ~600 warm-up + 1200 effective
    destroy_ratio_min:   float = 0.10
    destroy_ratio_max:   float = 0.40
    temp_control:        float = 0.05
    temp_decay:          float = 0.99975
    sigma1:              int   = 33
    sigma2:              int   = 9
    sigma3:              int   = 3
    weight_decay:        float = 0.10
    segment_size:        int   = 50
    early_stop_patience: int   = 400

    # ── DQN (constructive) ────────────────────────────────────────────────────
    dqn_state_dim:       int   = 13
    dqn_hidden:          int   = 128
    dqn_lr:              float = 1e-3
    dqn_gamma:           float = 0.95
    dqn_eps_start:       float = 0.80
    dqn_eps_end:         float = 0.05
    dqn_eps_decay:       float = 0.997
    dqn_buffer:          int   = 8_000
    dqn_batch:           int   = 64
    dqn_target_freq:     int   = 20
    dqn_train_freq:      int   = 5
    dqn_vehicle_penalty: float = 10.0

    # ── RL-ALNS ───────────────────────────────────────────────────────────────
    rla_state_dim:       int   = 14
    rla_hidden:          int   = 128
    rla_lr:              float = 1e-3
    rla_gamma:           float = 0.95
    rla_eps_start:       float = 0.40
    rla_eps_end:         float = 0.01
    rla_eps_decay:       float = 0.9997
    rla_buffer:          int   = 8_000
    rla_batch:           int   = 64
    rla_target_freq:     int   = 200
    rla_train_freq:      int   = 10
    rla_prefill:         int   = 300   # random episodes to fill buffer before training
    rla_reward_vehicle:  float = 15.0   # v5: NV-first (was 5.0)
    rla_reward_nv_inc:   float = -8.0   # v5: penalty for NV increase
    rla_reward_cost:     float = 1.0
    rla_reward_bonus:    float = 2.0
    rla_reward_bad:      float = -5.0   # v5: stronger infeasible penalty
    rla_gate_threshold:  float = 0.05   # v5: Q spread < this → defer to ALNS

    # ── Experiment ────────────────────────────────────────────────────────────
    n_runs:              int   = 5
    seed:                int   = 42

CFG = Config()

BKS = {
    'RC101': {'nv': 14, 'td': 1696.94}, 'RC102': {'nv': 12, 'td': 1554.75},
    'RC103': {'nv': 11, 'td': 1261.67}, 'RC104': {'nv': 10, 'td': 1135.48},
    'RC105': {'nv': 13, 'td': 1629.44}, 'RC106': {'nv': 11, 'td': 1424.73},
    'RC107': {'nv': 11, 'td': 1230.48}, 'RC108': {'nv': 10, 'td': 1139.82},
    'RC201': {'nv': 4,  'td': 1406.94}, 'RC202': {'nv': 3,  'td': 1365.64},
    'RC203': {'nv': 3,  'td': 1049.62}, 'RC204': {'nv': 3,  'td': 798.46},
    'RC205': {'nv': 4,  'td': 1297.65}, 'RC206': {'nv': 3,  'td': 1146.32},
    'RC207': {'nv': 3,  'td': 1061.14}, 'RC208': {'nv': 3,  'td': 828.14},
}
print('✅ Config ready.')

✅ Config ready.


In [3]:
# ── 3. Data ───────────────────────────────────────────────────────────────────
class Inst:
    def __init__(self, raw: Dict):
        self.name         = raw['name']
        data              = raw['data']
        self.capacity     = raw['capacity']
        self.coords       = data[:, 1:3]
        self.demands      = data[:, 3]
        self.ready_times  = data[:, 4]
        self.due_times    = data[:, 5]
        self.service_times= data[:, 6]
        self.horizon      = self.due_times[0]
        self.n            = len(data) - 1
        diff = self.coords[:, None, :] - self.coords[None, :, :]
        self.dist         = np.sqrt((diff**2).sum(axis=2))
        self.max_dist     = self.dist.max()
        self.tw_width     = self.due_times - self.ready_times
        self.max_tw_width = self.tw_width[1:].max() + 1e-9
        self.tw_tight_frac= sum(1 for i in range(1, self.n+1)
                                if self.tw_width[i] < 0.2*self.horizon) / self.n

def load_datasets(base: str) -> Dict:
    ds = {}
    for grp in ('rc1', 'rc2'):
        files = sorted(glob.glob(os.path.join(base, f'{grp}*.txt')))
        insts = []
        for path in files:
            with open(path) as f: lines = f.readlines()
            name     = lines[0].strip()
            capacity = float(lines[4].strip().split()[1])
            rows     = [list(map(float, l.split())) for l in lines[9:] if l.strip()]
            insts.append(Inst({'name': name, 'capacity': capacity, 'data': np.array(rows)}))
        ds[grp] = insts
        print(f'  {grp.upper()}: {len(insts)} instances')
    return ds

print('Loading...')
DATASETS = load_datasets(DATA_PATH)
RC1 = DATASETS['rc1']; RC2 = DATASETS['rc2']
print(f'✅ {RC1[0].name}: n={RC1[0].n}, tight_TW={RC1[0].tw_tight_frac:.0%}')

Loading...
  RC1: 8 instances
  RC2: 8 instances
✅ RC101: n=100, tight_TW=100%


In [4]:
# ── 4. Solution Model ─────────────────────────────────────────────────────────
@njit(cache=True)
def _rcost(route, dist):
    c = dist[0, route[0]]
    for i in range(len(route)-1): c += dist[route[i], route[i+1]]
    return c + dist[route[-1], 0]

@njit(cache=True)
def _rok(route, demands, cap, ready, due, service, dist):
    load = 0.0
    for n in route: load += demands[n]
    if load > cap: return False
    t, prev = 0.0, 0
    for n in route:
        t += dist[prev, n]
        if t < ready[n]: t = ready[n]
        if t > due[n]:   return False
        t += service[n]; prev = n
    return True

class Plan:
    __slots__ = ('routes', 'inst', '_cost', '_ok', 'algo')
    def __init__(self, routes, inst, algo=''):
        self.routes = [r for r in routes if r]
        self.inst   = inst; self._cost = None; self._ok = None; self.algo = algo

    @property
    def cost(self):
        if self._cost is None:
            self._cost = sum(_rcost(np.array(r, np.int64), self.inst.dist) for r in self.routes)
        return self._cost

    @property
    def feasible(self):
        if self._ok is None:
            inst = self.inst
            self._ok = all(_rok(np.array(r, np.int64), inst.demands, inst.capacity,
                               inst.ready_times, inst.due_times, inst.service_times, inst.dist)
                           for r in self.routes)
        return self._ok

    @property
    def nv(self): return len(self.routes)

    def dominates(self, o):
        return self.nv < o.nv or (self.nv == o.nv and self.cost < o.cost)

    def copy(self): return Plan([r[:] for r in self.routes], self.inst, self.algo)
    def inv(self):  self._cost = self._ok = None

    @property
    def on_time_rate(self):
        inst = self.inst; on = total = 0
        for route in self.routes:
            t, prev = 0.0, 0
            for n in route:
                t += inst.dist[prev, n]; t = max(t, inst.ready_times[n])
                total += 1
                if t <= inst.due_times[n]: on += 1
                t += inst.service_times[n]; prev = n
        return on / max(total, 1)

    def gap(self):
        bks = BKS.get(self.inst.name, {})
        td  = (self.cost-bks['td'])/bks['td']*100 if bks else None
        nv  = self.nv - bks['nv']                  if bks else None
        return td, nv

    def __repr__(self):
        td, nv = self.gap(); g = f'TD {td:+.1f}% NV {nv:+d}' if td else ''
        return f'Plan({self.algo}, nv={self.nv}, cost={self.cost:.1f}, {g})'

print('✅ Solution model ready.')

✅ Solution model ready.


In [5]:
# ── 5. ALNS Operators ─────────────────────────────────────────────────────────
def _inv(p): p.inv(); return p

def op_random(p, sz):
    nodes = [n for r in p.routes for n in r]; rem = random.sample(nodes, min(sz, len(nodes)))
    s = set(rem); p.routes = [[n for n in r if n not in s] for r in p.routes]
    p.routes = [r for r in p.routes if r]; return _inv(p), rem

def op_worst(p, sz):
    inst = p.inst; gain = []
    for route in p.routes:
        for i, n in enumerate(route):
            prev = route[i-1] if i>0 else 0; nx = route[i+1] if i<len(route)-1 else 0
            gain.append((inst.dist[prev,n]+inst.dist[n,nx]-inst.dist[prev,nx], n))
    gain.sort(reverse=True); rem = [n for _,n in gain[:sz]]
    s = set(rem); p.routes = [[n for n in r if n not in s] for r in p.routes]
    p.routes = [r for r in p.routes if r]; return _inv(p), rem

def op_shaw(p, sz):
    inst = p.inst; nodes = [n for r in p.routes for n in r]
    if not nodes: return p, []
    seed = random.choice(nodes); rem = [seed]; rs = {seed}
    md = inst.max_dist+1e-9; mt = max(inst.due_times-inst.ready_times)+1e-9
    while len(rem) < sz:
        cands = [(n, 0.5*inst.dist[seed,n]/md
                     + 0.4*abs(inst.ready_times[seed]-inst.ready_times[n])/mt
                     + 0.1*abs(inst.demands[seed]-inst.demands[n])/inst.capacity)
                 for n in nodes if n not in rs]
        if not cands: break
        rem.append(min(cands, key=lambda x: x[1])[0]); rs.add(rem[-1])
    s = set(rem); p.routes = [[n for n in r if n not in s] for r in p.routes]
    p.routes = [r for r in p.routes if r]; return _inv(p), rem

def op_route(p, sz):
    if len(p.routes) <= 1: return op_random(p, sz)
    rem, to_rm = [], set()
    for idx, route in sorted(enumerate(p.routes), key=lambda x: len(x[1])):
        if len(rem)+len(route) <= sz*1.5: rem.extend(route); to_rm.add(idx)
        if len(rem) >= sz: break
    p.routes = [r for i,r in enumerate(p.routes) if i not in to_rm] or [[]]
    return _inv(p), rem

def op_tw_urgent(p, sz):
    """Remove tightest-TW stops — forces re-routing of SLA-critical orders."""
    inst = p.inst; nodes = [n for r in p.routes for n in r]
    if not nodes: return p, []
    rem = sorted(nodes, key=lambda n: inst.due_times[n]-inst.ready_times[n])[:sz]
    s = set(rem); p.routes = [[n for n in r if n not in s] for r in p.routes]
    p.routes = [r for r in p.routes if r]; return _inv(p), rem

def _chk(route, inst):
    return bool(_rok(np.array(route, np.int64), inst.demands, inst.capacity,
                     inst.ready_times, inst.due_times, inst.service_times, inst.dist))

def _bestpos(n, route, inst):
    bc, bp = float('inf'), None
    for p in range(len(route)+1):
        pv = route[p-1] if p>0 else 0; nx = route[p] if p<len(route) else 0
        c = inst.dist[pv,n]+inst.dist[n,nx]-inst.dist[pv,nx]
        if c < bc and _chk(route[:p]+[n]+route[p:], inst): bc, bp = c, p
    return bc, bp

def _ins(plan, n, inst):
    bc, br, bp = float('inf'), None, None
    for ri, route in enumerate(plan.routes):
        c, p = _bestpos(n, route, inst)
        if p is not None and c < bc: bc, br, bp = c, ri, p
    if br is not None: plan.routes[br].insert(bp, n)
    else:              plan.routes.append([n])
    plan.inv()

def op_greedy(plan, rem):
    inst = plan.inst
    for n in sorted(rem, key=lambda n: inst.due_times[n]): _ins(plan, n, inst)
    return Plan(plan.routes, inst, plan.algo)

def _regret(plan, rem, k):
    inst = plan.inst; rem = rem[:]
    while rem:
        br, chosen, ci = -float('inf'), None, None
        for n in rem:
            opts = sorted((c,ri,p) for ri,r in enumerate(plan.routes)
                          for c,p in [_bestpos(n,r,inst)] if p is not None)
            reg = (sum(opts[i][0]-opts[0][0] for i in range(1,k)) if len(opts)>=k else
                   (opts[1][0]-opts[0][0] if len(opts)>=2 else
                    (float('inf') if len(opts)==1 else -float('inf'))))
            if reg > br and opts: br, chosen, ci = reg, n, opts[0]
        if chosen is not None:
            _, ri, p = ci; plan.routes[ri].insert(p, chosen); plan.inv(); rem.remove(chosen)
        else:
            for n in rem: plan.routes.append([n]); break
    return Plan(plan.routes, inst, plan.algo)

def op_reg2(p, rem): return _regret(p, rem, 2)
def op_reg3(p, rem): return _regret(p, rem, 3)

def op_tw_greedy(plan, rem):
    """Insert tightest-window orders first."""
    inst = plan.inst
    for n in sorted(rem, key=lambda n: inst.due_times[n]-inst.ready_times[n]):
        _ins(plan, n, inst)
    return Plan(plan.routes, inst, plan.algo)

DESTROY = [op_random, op_worst, op_shaw, op_route, op_tw_urgent]
REPAIR  = [op_greedy, op_reg2, op_reg3, op_tw_greedy]
N_D, N_R = len(DESTROY), len(REPAIR)
print(f'✅ Operators: {N_D}D × {N_R}R = {N_D*N_R} combos')

✅ Operators: 5D × 4R = 20 combos


In [6]:
# ── 6. Helpers ────────────────────────────────────────────────────────────────
def build_greedy(inst, algo='') -> Plan:
    customers = sorted(range(1, inst.n+1), key=lambda n: (inst.due_times[n], inst.ready_times[n]))
    unvisited = set(customers); routes = []
    while unvisited:
        route, node, load, t = [], 0, 0.0, 0.0
        while unvisited:
            feasible = [n for n in unvisited
                        if load+inst.demands[n] <= inst.capacity
                        and t+inst.dist[node,n] <= inst.due_times[n]]
            if not feasible: break
            nxt = min(feasible, key=lambda n: inst.dist[node,n])
            route.append(nxt); unvisited.remove(nxt)
            load += inst.demands[nxt]
            t = max(t+inst.dist[node,nxt], inst.ready_times[nxt]) + inst.service_times[nxt]
            node = nxt
        if route: routes.append(route)
    return Plan(routes, inst, algo)

def accept(cur: Plan, cand: Plan, temp: float) -> bool:
    if not cand.feasible: return False
    if cand.nv < cur.nv:  return True
    if cand.nv == cur.nv:
        if cand.cost < cur.cost: return True
        return random.random() < math.exp(-(cand.cost-cur.cost)/max(temp, 1e-6))
    return False

def dsz(it, n_iters, cfg: Config, n: int) -> int:
    ratio = cfg.destroy_ratio_max - (cfg.destroy_ratio_max-cfg.destroy_ratio_min)*(it/n_iters)
    return max(3, int(ratio*n))

print('✅ Helpers ready.')

✅ Helpers ready.


In [7]:
# ── 7. Neural Networks ────────────────────────────────────────────────────────
class DQNNet(nn.Module):
    def __init__(self, sd, ad, h):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(sd, h), nn.LayerNorm(h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(), nn.Linear(h, ad))
    def forward(self, x): return self.net(x)

class DuelingNet(nn.Module):
    """Dueling DQN: Q(s,a) = V(s) + A(s,a) - mean(A). Reduces overestimation."""
    def __init__(self, sd, ad, h):
        super().__init__()
        self.feat = nn.Sequential(nn.Linear(sd, h), nn.LayerNorm(h), nn.ReLU())
        self.val  = nn.Sequential(nn.Linear(h, h//2), nn.ReLU(), nn.Linear(h//2, 1))
        self.adv  = nn.Sequential(nn.Linear(h, h//2), nn.ReLU(), nn.Linear(h//2, ad))
    def forward(self, x):
        f = self.feat(x); v = self.val(f); a = self.adv(f)
        return v + a - a.mean(dim=-1, keepdim=True)

class ReplayBuffer:
    def __init__(self, cap): self.buf = deque(maxlen=cap)
    def push(self, *a): self.buf.append(a)
    def sample(self, k):
        s,a,r,ns,d = zip(*random.sample(self.buf, k))
        return (np.array(s, np.float32), np.array(a, np.int64),
                np.array(r, np.float32), np.array(ns, np.float32), np.array(d, np.float32))
    def __len__(self): return len(self.buf)

print('✅ Networks ready.')

✅ Networks ready.


In [8]:
# ── 8. ALNS Solver ───────────────────────────────────────────────────────────
class ALNSSolver:
    def __init__(self, inst: Inst, cfg: Config = CFG):
        self.inst = inst; self.cfg = cfg

    def _roulette(self, w): return int(np.random.choice(len(w), p=w/w.sum()))

    def solve(self, init: Plan = None, seed=None) -> Tuple[Plan, List[float]]:
        if seed is not None: random.seed(seed); np.random.seed(seed)
        cfg = self.cfg
        cur = init.copy() if init else build_greedy(self.inst, 'ALNS'); cur.algo='ALNS'
        best = cur.copy(); temp = cfg.temp_control*cur.cost/math.log(2)
        dw = np.ones(N_D); rw = np.ones(N_R)
        ss = np.zeros((N_D,N_R)); sc = np.zeros((N_D,N_R))
        hist = [best.cost]; no_imp = 0

        for it in range(cfg.alns_iterations):
            di = self._roulette(dw); ri = self._roulette(rw)
            sz = dsz(it, cfg.alns_iterations, cfg, self.inst.n)
            dest, rem = DESTROY[di](cur.copy(), sz); cand = REPAIR[ri](dest, rem)
            score = 0
            if accept(cur, cand, temp):
                if cand.dominates(best):  best=cand.copy(); score=cfg.sigma1; no_imp=0
                elif cand.dominates(cur): score=cfg.sigma2; no_imp=0
                else:                     score=cfg.sigma3; no_imp+=1
                cur = cand
            else: no_imp += 1
            ss[di,ri]+=score; sc[di,ri]+=1
            if (it+1)%cfg.segment_size==0:
                for d in range(N_D):
                    for r in range(N_R):
                        if sc[d,r]>0:
                            avg=ss[d,r]/sc[d,r]
                            dw[d]=dw[d]*(1-cfg.weight_decay)+avg*cfg.weight_decay
                            rw[r]=rw[r]*(1-cfg.weight_decay)+avg*cfg.weight_decay
                ss[:]=0; sc[:]=0; dw=np.maximum(dw,0.1); rw=np.maximum(rw,0.1)
            temp*=cfg.temp_decay; hist.append(best.cost)
            if no_imp>=cfg.early_stop_patience: break
        best.algo='ALNS'; return best, hist

print('✅ ALNS ready.')

✅ ALNS ready.


In [9]:
# ── 9. DQN Solver (ablation) ──────────────────────────────────────────────────
class DQNSolver:
    def __init__(self, inst: Inst, cfg: Config = CFG):
        self.inst = inst; self.cfg = cfg
        self.q   = DQNNet(cfg.dqn_state_dim, inst.n+1, cfg.dqn_hidden).to(DEVICE)
        self.q_t = DQNNet(cfg.dqn_state_dim, inst.n+1, cfg.dqn_hidden).to(DEVICE)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt = optim.Adam(self.q.parameters(), lr=cfg.dqn_lr)
        self.buf = ReplayBuffer(cfg.dqn_buffer); self.eps = cfg.dqn_eps_start

    def _state(self, node, visited, load, t):
        inst = self.inst; uv = inst.n - len(visited)
        feas = [n for n in range(1, inst.n+1)
                if n not in visited and load+inst.demands[n]<=inst.capacity
                and t+inst.dist[node,n]<=inst.due_times[n]]
        nf = len(feas)
        if feas:
            slacks = [inst.due_times[n]-(t+inst.dist[node,n]) for n in feas]
            ms=min(slacks)/max(inst.horizon,1); av=(sum(slacks)/nf)/max(inst.horizon,1)
            uf=sum(1 for s in slacks if s<0.1*inst.horizon)/max(nf,1)
            aw=(sum(inst.tw_width[n] for n in feas)/nf)/max(inst.max_tw_width,1)
        else: ms=av=uf=aw=0.0
        return np.array([load/inst.capacity, t/max(inst.horizon,1),
                         len(visited)/inst.n, (inst.capacity-load)/inst.capacity,
                         uv/inst.n, nf/max(uv,1),
                         inst.coords[node,0]/100, inst.coords[node,1]/100,
                         inst.demands[node]/inst.capacity,
                         ms, av, uf, aw], dtype=np.float32)

    def _acts(self, node, visited, load, t):
        inst = self.inst; acts = [0]
        for n in range(1, inst.n+1):
            if (n not in visited and load+inst.demands[n]<=inst.capacity
                    and t+inst.dist[node,n]<=inst.due_times[n]): acts.append(n)
        return acts

    def _sel(self, state, feasible):
        if random.random()<self.eps: return random.choice(feasible)
        with torch.no_grad():
            q = self.q(torch.tensor(state).unsqueeze(0).to(DEVICE)).cpu().numpy()[0]
        return max(feasible, key=lambda a: q[a])

    def _train(self):
        if len(self.buf)<self.cfg.dqn_batch: return
        s,a,r,ns,d = self.buf.sample(self.cfg.dqn_batch)
        s=torch.tensor(s).to(DEVICE); a=torch.tensor(a,dtype=torch.long).to(DEVICE)
        r=torch.tensor(r).to(DEVICE); ns=torch.tensor(ns).to(DEVICE); d=torch.tensor(d).to(DEVICE)
        qp=self.q(s).gather(1,a.unsqueeze(1)).squeeze(1)
        with torch.no_grad(): tgt=r+self.cfg.dqn_gamma*self.q_t(ns).max(1)[0]*(1-d)
        loss=F.mse_loss(qp,tgt); self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(),1.0); self.opt.step()

    def _episode(self):
        inst=self.inst; visited=set(); routes=[]; trans=[]
        while len(visited)<inst.n:
            route,node,load,t,is_new = [],0,0.0,0.0,True
            while True:
                state=self._state(node,visited,load,t); feas=self._acts(node,visited,load,t)
                if len(feas)==1: break
                action=self._sel(state,feas)
                if action==0: break
                dv=inst.dist[node,action]
                rew=-dv/max(inst.max_dist,1)-(self.cfg.dqn_vehicle_penalty/inst.n if is_new and routes else 0)
                is_new=False; load+=inst.demands[action]
                t=max(t+dv,inst.ready_times[action])+inst.service_times[action]
                visited.add(action); route.append(action)
                ns=self._state(action,visited,load,t); done=(len(visited)==inst.n)
                trans.append((state,action,rew,ns,float(done))); node=action
            if route: routes.append(route)
        return Plan(routes,inst,'DQN'), trans

    def solve(self, seed=None) -> Tuple[Plan, List[float]]:
        if seed is not None: random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        cfg=self.cfg; best=None; bc=float('inf'); hist=[]; self.eps=cfg.dqn_eps_start
        for ep in range(max(50, cfg.alns_iterations//self.inst.n)):
            plan,trans=self._episode()
            if plan.feasible and trans:
                bonus=max(0,(bc-plan.cost)/bc*10) if bc<float('inf') else 1.0
                s,a,r,ns,d=trans[-1]; trans[-1]=(s,a,r+bonus,ns,d)
                if plan.cost<bc: bc=plan.cost; best=plan.copy()
            for tr in trans: self.buf.push(*tr)
            if ep%cfg.dqn_train_freq==0:
                for _ in range(min(5,len(self.buf)//max(cfg.dqn_batch,1))): self._train()
            if ep%cfg.dqn_target_freq==0: self.q_t.load_state_dict(self.q.state_dict())
            self.eps=max(cfg.dqn_eps_end,self.eps*cfg.dqn_eps_decay)
            hist.append(bc if bc<float('inf') else float('nan'))
        if best is None: best=build_greedy(self.inst,'DQN')
        best.algo='DQN'; return best, hist

    def save(self, path): save_file({k:v.cpu() for k,v in self.q.state_dict().items()}, path)
    def load(self, path): self.q.load_state_dict(load_file(path)); self.q_t.load_state_dict(self.q.state_dict())

print('✅ DQN solver ready.')

✅ DQN solver ready.


In [10]:
# ── 10. RL-ALNS Solver — Main Contribution ───────────────────────────────────
#
# Key design decisions vs vanilla ALNS:
#   1. Dueling Double DQN → less overestimation in sparse reward landscape
#   2. 14D normalized state (incl. TW slack + instance difficulty)
#   3. Hierarchical reward: Δvehicles > Δcost (matches VRPTW objective)
#   4. Pre-fill buffer with N_PREFILL random episodes before training starts
#      → RL gets useful signal immediately, not wasted on empty buffer
#   5. rla_iterations > alns_iterations: accounts for warm-up phase
#   6. Transfer-ready: save/load weights for cross-instance generalisation

class RLALNSSolver:
    def __init__(self, inst: Inst, cfg: Config = CFG):
        self.inst = inst; self.cfg = cfg
        self.q   = DuelingNet(cfg.rla_state_dim, N_D*N_R, cfg.rla_hidden).to(DEVICE)
        self.q_t = DuelingNet(cfg.rla_state_dim, N_D*N_R, cfg.rla_hidden).to(DEVICE)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt = optim.Adam(self.q.parameters(), lr=cfg.rla_lr)
        self.buf = ReplayBuffer(cfg.rla_buffer); self.eps = cfg.rla_eps_start
        self.op_counts: Dict = {}  # v5: track operator usage

    def _state(self, cur: Plan, best: Plan, it, n_iters, temp, dw, rw, imps):
        inst = self.inst; cfg = self.cfg
        imp_rate  = sum(imps)/len(imps) if imps else 0.0
        cost_gap  = min((cur.cost-best.cost)/max(best.cost,1), 1.0)
        nv_ratio  = cur.nv/max(self._init_nv,1)
        progress  = it/n_iters
        lens  = [len(r) for r in cur.routes] or [0]
        loads = [sum(inst.demands[n] for n in r) for r in cur.routes] or [0]
        rb = float(np.std(lens)/max(np.mean(lens),1)) if len(lens)>1 else 0.0
        lb = float(np.std(loads)/max(inst.capacity,1))
        T0 = cfg.temp_control*best.cost/math.log(2)
        tn = min(temp/max(T0,1e-6),1.0)
        dp = dw/dw.sum(); rp = rw/rw.sum()
        # TW features
        tw_tight = inst.tw_tight_frac
        tw_slack = self._avg_slack(cur)
        return np.array([cost_gap, nv_ratio, progress, imp_rate, 1-imp_rate,
                         min(rb,1.0), min(lb,1.0), tn,
                         dp.max(), dp.min(), rp.max(), rp.min(),
                         tw_tight, tw_slack], dtype=np.float32)

    def _avg_slack(self, plan: Plan) -> float:
        inst=plan.inst; sl=0.0; n=0
        for route in plan.routes:
            t,prev=0.0,0
            for nd in route:
                t+=inst.dist[prev,nd]; t=max(t,inst.ready_times[nd])
                sl+=inst.due_times[nd]-t; t+=inst.service_times[nd]; prev=nd; n+=1
        return (sl/n)/max(inst.horizon,1) if n else 0.0

    def _act(self, state, dw=None, rw=None):
        """v5: defer to ALNS weights when RL confidence is low."""
        if random.random() < self.eps:
            return random.randrange(N_D*N_R)
        with torch.no_grad():
            q = self.q(torch.tensor(state).unsqueeze(0).to(DEVICE)).cpu().numpy()[0]
        confidence = float(q.max() - q.mean())
        if confidence < self.cfg.rla_gate_threshold and dw is not None:
            di = int(np.random.choice(N_D, p=dw/dw.sum()))
            ri = int(np.random.choice(N_R, p=rw/rw.sum()))
            return di * N_R + ri
        return int(q.argmax())

    def _reward(self, cur: Plan, cand: Plan, best: Plan) -> float:
        """v5: NV-first — vehicle reduction dominates, penalty for NV increase."""
        cfg = self.cfg
        if not cand.feasible: return cfg.rla_reward_bad
        nv_delta = cur.nv - cand.nv
        r = nv_delta * cfg.rla_reward_vehicle
        if nv_delta < 0:  # NV increased → extra penalty
            r += cfg.rla_reward_nv_inc
        r += (cur.cost-cand.cost)/max(cur.cost,1)*100*cfg.rla_reward_cost
        if cand.dominates(best): r += cfg.rla_reward_bonus
        return float(r)

    def _train(self):
        if len(self.buf)<self.cfg.rla_batch: return
        s,a,r,ns,d = self.buf.sample(self.cfg.rla_batch)
        s=torch.tensor(s).to(DEVICE); a=torch.tensor(a,dtype=torch.long).to(DEVICE)
        r=torch.tensor(r).to(DEVICE); ns=torch.tensor(ns).to(DEVICE); d=torch.tensor(d).to(DEVICE)
        qp=self.q(s).gather(1,a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():  # Double DQN
            an=self.q(ns).argmax(1)
            qn=self.q_t(ns).gather(1,an.unsqueeze(1)).squeeze(1)
            tgt=r+self.cfg.rla_gamma*qn*(1-d)
        loss=F.mse_loss(qp,tgt); self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(),1.0); self.opt.step()

    def _prefill(self, n_episodes: int):
        """
        Pre-fill replay buffer with random-operator episodes.
        Ensures RL has meaningful training signal from iteration 1.
        """
        cur = build_greedy(self.inst); best = cur.copy()
        self._init_nv = cur.nv
        temp = self.cfg.temp_control*cur.cost/math.log(2)
        dw = np.ones(N_D); rw = np.ones(N_R)
        imps = deque(maxlen=50)
        for ep in range(n_episodes):
            it = ep % max(self.cfg.rla_iterations, 1)
            state = self._state(cur, best, it, self.cfg.rla_iterations, temp, dw, rw, imps)
            action = random.randrange(N_D*N_R)  # fully random
            di, ri = action//N_R, action%N_R
            sz = dsz(it, self.cfg.rla_iterations, self.cfg, self.inst.n)
            dest, rem = DESTROY[di](cur.copy(), sz); cand = REPAIR[ri](dest, rem)
            reward = self._reward(cur, cand, best)
            if accept(cur, cand, temp):
                if cand.dominates(best): best=cand.copy()
                cur=cand; imps.append(1)
            else: imps.append(0)
            temp *= self.cfg.temp_decay
            ns_state = self._state(cur, best, it+1, self.cfg.rla_iterations, temp, dw, rw, imps)
            self.buf.push(state, action, reward, ns_state, 0.0)

    def solve(self, seed=None, frozen=False) -> Tuple[Plan, List[float]]:
        """
        frozen=True: don't update weights (inference / transfer mode).
        """
        if seed is not None: random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        cfg = self.cfg; n_iters = cfg.rla_iterations
        cur = build_greedy(self.inst, 'RL-ALNS'); best = cur.copy()
        self._init_nv = cur.nv
        temp = cfg.temp_control*cur.cost/math.log(2)
        dw = np.ones(N_D); rw = np.ones(N_R)
        ss = np.zeros((N_D,N_R)); sc = np.zeros((N_D,N_R))
        imps = deque(maxlen=50); hist=[best.cost]; no_imp=0
        if not frozen: self.eps = cfg.rla_eps_start; self.op_counts = {}

        # Pre-fill buffer before search starts
        if not frozen and len(self.buf) < cfg.rla_prefill:
            self._prefill(cfg.rla_prefill - len(self.buf))

        for it in range(n_iters):
            state = self._state(cur, best, it, n_iters, temp, dw, rw, imps)
            action = self._act(state, dw, rw); di, ri = action//N_R, action%N_R
            sz = dsz(it, n_iters, cfg, self.inst.n)
            dest, rem = DESTROY[di](cur.copy(), sz); cand = REPAIR[ri](dest, rem)
            reward = self._reward(cur, cand, best); ascore = 0

            if accept(cur, cand, temp):
                improved = cand.dominates(cur); imps.append(1 if improved else 0)
                if cand.dominates(best):  best=cand.copy(); ascore=cfg.sigma1; no_imp=0
                elif improved:            ascore=cfg.sigma2; no_imp=0
                else:                     ascore=cfg.sigma3; no_imp+=1
                cur=cand
            else: imps.append(0); no_imp+=1

            ss[di,ri]+=ascore; sc[di,ri]+=1
            self.op_counts[(di,ri)] = self.op_counts.get((di,ri), 0) + 1
            if (it+1)%cfg.segment_size==0:
                for d in range(N_D):
                    for r in range(N_R):
                        if sc[d,r]>0:
                            avg=ss[d,r]/sc[d,r]
                            dw[d]=dw[d]*(1-cfg.weight_decay)+avg*cfg.weight_decay
                            rw[r]=rw[r]*(1-cfg.weight_decay)+avg*cfg.weight_decay
                ss[:]=0; sc[:]=0; dw=np.maximum(dw,0.1); rw=np.maximum(rw,0.1)

            ns = self._state(cur, best, it+1, n_iters, temp, dw, rw, imps)
            self.buf.push(state, action, reward, ns, float(it==n_iters-1))
            if not frozen:
                if it%cfg.rla_train_freq ==0: self._train()
                if it%cfg.rla_target_freq==0: self.q_t.load_state_dict(self.q.state_dict())
                self.eps = max(cfg.rla_eps_end, self.eps*cfg.rla_eps_decay)
            temp*=cfg.temp_decay; hist.append(best.cost)
            if no_imp>=cfg.early_stop_patience: break

        best.algo = 'RL-ALNS'; return best, hist

    def save(self, path): save_file({k:v.cpu() for k,v in self.q.state_dict().items()}, path)
    def load(self, path):
        self.q.load_state_dict(load_file(path)); self.q_t.load_state_dict(self.q.state_dict())

    def clone_weights(self) -> Dict:
        """Return a copy of current weights (for transfer)."""
        return {k: v.clone().cpu() for k, v in self.q.state_dict().items()}

    def set_weights(self, weights: Dict):
        """Load weights from another instance (transfer learning)."""
        self.q.load_state_dict({k: v.to(DEVICE) for k, v in weights.items()})
        self.q_t.load_state_dict(self.q.state_dict())

print('✅ RL-ALNS ready.')

✅ RL-ALNS ready.


In [11]:
# ── 11. Benchmark Runner ──────────────────────────────────────────────────────
def run_instance(inst: Inst, algo: str, cfg: Config, seed: int,
                 transfer_weights: Dict = None) -> Dict:
    """Run one algorithm on one instance with one seed. Returns result dict."""
    t0 = time.time()
    if algo == 'ALNS':
        plan, hist = ALNSSolver(inst, cfg).solve(seed=seed)
    elif algo == 'DQN':
        plan, hist = DQNSolver(inst, cfg).solve(seed=seed)
    elif algo == 'RL-ALNS':
        solver = RLALNSSolver(inst, cfg)
        plan, hist = solver.solve(seed=seed)
    elif algo == 'RL-ALNS★':  # Transfer: load pre-trained weights, run frozen
        solver = RLALNSSolver(inst, cfg)
        if transfer_weights is not None:
            solver.set_weights(transfer_weights)
            solver.eps = cfg.rla_eps_end  # greedy — no more exploration
        plan, hist = solver.solve(seed=seed, frozen=True)
        plan.algo = 'RL-ALNS★'
    else:
        raise ValueError(algo)
    elapsed = time.time() - t0
    bks = BKS.get(inst.name, {})
    return {
        'nv': plan.nv, 'cost': plan.cost, 'time': elapsed,
        'td_gap': (plan.cost-bks['td'])/bks['td']*100 if bks.get('td') else None,
        'nv_diff': plan.nv - bks['nv']                 if bks.get('nv') else None,
        'on_time': plan.on_time_rate,
        'hist': hist,
    }


def run_benchmark(instances: List[Inst],
                  algorithms: List[str],
                  cfg: Config = CFG,
                  result_path: str = None,
                  transfer_weights: Dict = None) -> pd.DataFrame:
    if result_path is None:
        result_path = os.path.join(OUTPUT_DIR, 'benchmark.csv')

    done = set()
    if os.path.exists(result_path):
        ex = pd.read_csv(result_path)
        done = set(zip(ex['Instance'], ex['Algorithm']))
        print(f'  Resuming — {len(done)} combos done')

    total = len(instances)*len(algorithms)
    print(f'  Total: {total} combos × {cfg.n_runs} runs\n' + '='*60)

    for inst in instances:
        ds = 'RC1' if int(inst.name[2]) == 1 else 'RC2'
        for algo in algorithms:
            if (inst.name, algo) in done: print(f'  ⏭ {inst.name} {algo}'); continue
            print(f'\n  [{inst.name}] {algo}')
            rows_nv, rows_cost, rows_time, rows_gap, rows_nv_diff, rows_ot = [], [], [], [], [], []

            for run in range(cfg.n_runs):
                res = run_instance(inst, algo, cfg, cfg.seed+run, transfer_weights)
                rows_nv.append(res['nv']); rows_cost.append(res['cost'])
                rows_time.append(res['time']); rows_gap.append(res['td_gap'])
                rows_nv_diff.append(res['nv_diff']); rows_ot.append(res['on_time'])
                print(f'    run {run+1}/{cfg.n_runs}: nv={res["nv"]} '
                      f'cost={res["cost"]:.1f} ({res["time"]:.1f}s)')

            row = pd.DataFrame([{
                'Dataset': ds, 'Instance': inst.name, 'Algorithm': algo,
                'NV_mean':   round(np.mean(rows_nv),2),
                'NV_std':    round(np.std(rows_nv), 2),
                'NV_diff':   round(np.mean(rows_nv_diff),2) if rows_nv_diff[0] is not None else None,
                'TD_mean':   round(np.mean(rows_cost),2),
                'TD_std':    round(np.std(rows_cost),2),
                'Gap%':      round(np.mean(rows_gap),2)     if rows_gap[0] is not None else None,
                'OnTime':    round(np.mean(rows_ot)*100,1),
                'Time_s':    round(np.mean(rows_time),1),
                'NV_cv':     round(np.std(rows_nv)/max(np.mean(rows_nv),1)*100,2),
                'TD_cv':     round(np.std(rows_cost)/max(np.mean(rows_cost),1)*100,2),
            }])
            row.to_csv(result_path, mode='a', header=not os.path.exists(result_path), index=False)
            g = rows_gap[0]; gv = f'{np.mean(rows_gap):+.1f}%' if g is not None else '—'
            print(f'  → nv={np.mean(rows_nv):.1f}±{np.std(rows_nv):.1f}  '
                  f'td={np.mean(rows_cost):.1f}±{np.std(rows_cost):.1f}  gap={gv}')

    return pd.read_csv(result_path)

print('✅ Benchmark runner ready.')

✅ Benchmark runner ready.


In [12]:
# ── 12. Transfer Learning Pipeline ───────────────────────────────────────────
#
# FLOW:
#   1. Train RL-ALNS on all RC1 instances (8 instances, n_runs each)
#      → RL learns: given state (cost gap, TW slack, operator weights, ...)
#        which operator pair tends to work best
#   2. Average the learned weights across all RC1 instances
#      → one generalised policy
#   3. Apply frozen policy to RC2 instances (zero-shot)
#      → compare RL-ALNS★ (transferred) vs ALNS (from scratch)
#
# This answers the research question:
#   "Can RL learn a transferable operator selection policy for VRPTW?"

def train_transfer_model(instances: List[Inst], cfg: Config = CFG,
                         seed: int = 42) -> Dict:
    """
    Train RL-ALNS on all instances in `instances`.
    Returns averaged Q-network weights (policy ensemble).
    """
    print('📚 Training transfer model...')
    all_weights = []

    for inst in instances:
        print(f'  Training on {inst.name}...')
        solver = RLALNSSolver(inst, cfg)
        plan, _ = solver.solve(seed=seed)
        all_weights.append(solver.clone_weights())
        td, nv = plan.gap()
        print(f'    nv={plan.nv}, gap={td:+.1f}%')

    # Average weights across all training instances
    keys = all_weights[0].keys()
    averaged = {k: torch.stack([w[k].float() for w in all_weights]).mean(0)
                for k in keys}

    # Save for reuse
    save_path = os.path.join(MODEL_DIR, 'rl_alns_transfer.safetensors')
    save_file(averaged, save_path)
    print(f'\n✅ Transfer model saved → {save_path}')
    return averaged


def load_transfer_model(cfg: Config = CFG) -> Optional[Dict]:
    path = os.path.join(MODEL_DIR, 'rl_alns_transfer.safetensors')
    if os.path.exists(path):
        print(f'✅ Transfer model loaded from {path}')
        return load_file(path)
    return None

print('✅ Transfer learning pipeline ready.')

✅ Transfer learning pipeline ready.


In [13]:
# ── 13. Statistical Tests & Paper Table ───────────────────────────────────────
def wilcoxon_test(df: pd.DataFrame, algo_a: str, algo_b: str,
                  metric: str = 'Gap%', dataset: str = None) -> Dict:
    """
    Wilcoxon signed-rank test: is algo_a significantly better than algo_b?
    Non-parametric (does not assume normality) — appropriate for n_instances=8.
    """
    sub = df if dataset is None else df[df['Dataset']==dataset]
    a = sub[sub['Algorithm']==algo_a][metric].dropna().values
    b = sub[sub['Algorithm']==algo_b][metric].dropna().values
    n = min(len(a), len(b)); a, b = a[:n], b[:n]
    if n < 3: return {'stat': None, 'p': None, 'sig': False, 'n': n}
    stat, p = stats.wilcoxon(a, b, alternative='two-sided')
    return {'stat': round(stat,3), 'p': round(p,4), 'sig': p<0.05, 'n': n,
            'better': algo_a if a.mean()<b.mean() else algo_b}


def print_paper_table(df: pd.DataFrame):
    """Full academic table with mean±std, Gap%, CV, on-time rate."""
    summary = (
        df.groupby(['Dataset','Algorithm'])
          .agg(NV=('NV_mean','mean'), NV_std=('NV_std','mean'),
               NV_d=('NV_diff','mean'),
               TD=('TD_mean','mean'), TD_std=('TD_std','mean'),
               Gap=('Gap%','mean'),
               CV_nv=('NV_cv','mean'), CV_td=('TD_cv','mean'),
               OT=('OnTime','mean'),
               Time=('Time_s','mean'))
          .round(2).reset_index()
    )
    cols = f'{"DS":<4}{"Algorithm":<12}{"NV":>6}{"±":>4}{"vs BKS":>8}{"TD":>9}{"±":>6}{"Gap%":>7}{"CV_NV":>6}{"CV_TD":>6}{"OT%":>6}{"Time":>7}'
    print('\n' + '─'*len(cols))
    print(cols)
    print('─'*len(cols))
    prev = ''
    for _, r in summary.iterrows():
        if r['Dataset'] != prev and prev: print('─'*len(cols))
        prev = r['Dataset']
        nv_d = f"{r['NV_d']:+.1f}" if pd.notna(r['NV_d']) else '—'
        gap  = f"{r['Gap']:+.1f}%" if pd.notna(r['Gap'])   else '—'
        print(f"{r['Dataset']:<4}{r['Algorithm']:<12}"
              f"{r['NV']:>6.1f}{r['NV_std']:>4.1f}{nv_d:>8}"
              f"{r['TD']:>9.1f}{r['TD_std']:>6.1f}{gap:>7}"
              f"{r['CV_nv']:>6.1f}{r['CV_td']:>6.1f}"
              f"{r['OT']:>6.1f}{r['Time']:>6.1f}s")
    print('─'*len(cols))
    print('CV = Coefficient of Variation (std/mean × 100%) — lower is more consistent.')
    print('Negative Gap% with NV_diff>0: extra vehicle allows shorter total routes.')


def print_stats_table(df: pd.DataFrame):
    """Wilcoxon test: RL-ALNS vs ALNS, per dataset."""
    print('\n── Statistical Significance (Wilcoxon signed-rank, two-sided) ──')
    print(f'{"Comparison":<28}{"Dataset":<6}{"Metric":<8}{"W-stat":>8}{"p-value":>9}{"Sig":>5}{"Better":>12}')
    print('─'*70)
    pairs = [('RL-ALNS','ALNS'), ('RL-ALNS★','ALNS')]
    for algo_a, algo_b in pairs:
        if algo_a not in df['Algorithm'].values: continue
        for ds in ['RC1','RC2']:
            for metric in ['Gap%','NV_mean']:
                res = wilcoxon_test(df, algo_a, algo_b, metric, ds)
                if res['stat'] is None: continue
                sig  = '✅' if res['sig'] else '—'
                print(f'  {algo_a} vs {algo_b:<8}  {ds:<6}{metric:<8}'
                      f'{res["stat"]:>8.1f}{res["p"]:>9.4f}{sig:>5}{res["better"]:>12}')
    print('─'*70)
    print('✅ = p < 0.05 (statistically significant difference)')

print('✅ Stats & table utilities ready.')

✅ Stats & table utilities ready.


In [14]:
# ── 14. Visualisation ─────────────────────────────────────────────────────────
COLORS = {'ALNS':'#5f5fae','DQN':'#e8593c','RL-ALNS':'#1d9e75','RL-ALNS★':'#f2a623'}
HATCH  = {'ALNS':'','DQN':'//','RL-ALNS':'','RL-ALNS★':'xx'}

def plot_dashboard(df: pd.DataFrame):
    """Main 2×3 dashboard: Gap%, NV, CV per dataset."""
    metrics = [
        ('Gap%',    'Distance Gap vs BKS (%)', '↓ lower is better'),
        ('NV_mean', 'Vehicles Used (avg)',      '↓ lower is better'),
        ('TD_cv',   'TD Consistency (CV %)',    '↓ lower = more stable'),
    ]
    algos = [a for a in COLORS if a in df['Algorithm'].values]
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    for ri, ds in enumerate(['RC1','RC2']):
        for ci, (met, label, note) in enumerate(metrics):
            ax  = axes[ri][ci]; sub = df[df['Dataset']==ds]
            insts = sub['Instance'].unique(); x = np.arange(len(insts)); w = 0.2
            for ji, algo in enumerate(algos):
                vals = [sub[sub['Instance']==i][met].mean() for i in insts]
                ax.bar(x+ji*w, vals, w, label=algo, color=COLORS[algo],
                       hatch=HATCH[algo], alpha=0.85, edgecolor='white')
            ax.set_xticks(x+w*1.5); ax.set_xticklabels([i[-3:] for i in insts], fontsize=8)
            ax.set_title(f'{ds} — {label}\n({note})', fontsize=9, fontweight='bold')
            ax.set_ylabel(met, fontsize=8); ax.grid(axis='y', alpha=0.3)
            if ri==0 and ci==0: ax.legend(fontsize=8)
    plt.suptitle('Algorithm Comparison Dashboard — VRPTW Solomon RC Benchmarks',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,'dashboard.png'), dpi=150, bbox_inches='tight')
    plt.show()


def plot_convergence_grid(inst: Inst, cfg: Config, seed: int = 42):
    """Run ALNS vs RL-ALNS on one instance, plot convergence side-by-side."""
    histories = {}
    for algo, Solver in [('ALNS', ALNSSolver), ('RL-ALNS', RLALNSSolver)]:
        s = Solver(inst, cfg)
        _, hist = s.solve(seed=seed)
        histories[algo] = hist

    fig, ax = plt.subplots(figsize=(9, 4))
    for algo, hist in histories.items():
        ax.plot(hist, label=algo, color=COLORS[algo], lw=2, alpha=0.9)
    bks = BKS.get(inst.name, {})
    if bks: ax.axhline(bks['td'], color='gray', ls='--', lw=1.2, label='BKS distance')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Best Cost Found')
    ax.set_title(f'Convergence — {inst.name}', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.2); plt.tight_layout(); plt.show()


def plot_transfer_comparison(df: pd.DataFrame):
    """Scatter: RL-ALNS★ gap vs ALNS gap per RC2 instance."""
    if 'RL-ALNS★' not in df['Algorithm'].values: return
    sub = df[df['Dataset']=='RC2']
    alns = sub[sub['Algorithm']=='ALNS'].set_index('Instance')['Gap%']
    rla  = sub[sub['Algorithm']=='RL-ALNS★'].set_index('Instance')['Gap%']
    common = alns.index.intersection(rla.index)

    fig, ax = plt.subplots(figsize=(7, 5))
    for inst_name in common:
        a, r = alns[inst_name], rla[inst_name]
        color = '#1d9e75' if r < a else '#e8593c'
        ax.scatter(a, r, color=color, s=80, zorder=3)
        ax.annotate(inst_name[-3:], (a, r), textcoords='offset points',
                    xytext=(5,3), fontsize=8)
    lo = min(alns[common].min(), rla[common].min()) - 1
    hi = max(alns[common].max(), rla[common].max()) + 1
    ax.plot([lo,hi],[lo,hi], 'k--', lw=1, label='y=x (equal)')
    ax.fill_between([lo,hi],[lo,lo],[hi,hi], alpha=0.05, color='green')
    ax.set_xlabel('ALNS Gap%'); ax.set_ylabel('RL-ALNS★ Gap% (transfer)')
    ax.set_title('RC2 Transfer: points below diagonal = RL-ALNS★ wins', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.2); plt.tight_layout(); plt.show()


def plot_routes(plan: Plan, figsize=(10,8)):
    RCOLS = ['#E63946','#2A9D8F','#E9C46A','#264653','#F4A261',
             '#A8DADC','#457B9D','#6A4C93','#F72585','#4CC9F0',
             '#80B918','#FF9F1C','#8338EC','#3A86FF','#CBFF8C']
    inst = plan.inst; fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(*inst.coords[0], s=220, c='black', marker='s', zorder=5)
    ax.annotate('DEPOT', inst.coords[0], fontsize=8, ha='center', va='bottom', fontweight='bold')
    for i, route in enumerate(plan.routes):
        col = RCOLS[i%len(RCOLS)]; stops = [0]+route+[0]
        xs = [inst.coords[n,0] for n in stops]; ys = [inst.coords[n,1] for n in stops]
        ax.plot(xs, ys, '-o', color=col, lw=1.5, ms=4, alpha=0.8, label=f'V{i+1}')
    td, nv = plan.gap(); g = f' | BKS: TD {td:+.1f}% NV {nv:+d}' if td else ''
    ax.set_title(f'{plan.algo} — {inst.name}  nv={plan.nv}  cost={plan.cost:.1f}{g}',
                 fontweight='bold')
    ax.legend(fontsize=6, ncol=3); ax.grid(alpha=0.2); plt.tight_layout(); plt.show()

print('✅ Visualisation ready.')

✅ Visualisation ready.


In [15]:
# ── 15. Smoke Test ────────────────────────────────────────────────────────────
smoke_cfg = Config(alns_iterations=400, rla_iterations=600,
                   early_stop_patience=150, n_runs=1, rla_prefill=100)
inst = RC1[0]; print(f'Smoke test — {inst.name}\n')
hists = {}

for algo, Solver in [('ALNS', ALNSSolver), ('DQN', DQNSolver), ('RL-ALNS', RLALNSSolver)]:
    t0 = time.time(); s = Solver(inst, smoke_cfg)
    plan, hist = s.solve(seed=42); el = time.time()-t0
    td, nv = plan.gap(); hists[algo] = hist
    print(f'  {algo:<12} nv={plan.nv:>3}  cost={plan.cost:>8.1f}  '
          f'BKS TD {td:+.1f}% NV {nv:+d}  ({el:.1f}s)')

print('\n✓ Smoke test passed')
plot_convergence_grid(RC1[0], smoke_cfg, seed=42)

Smoke test — RC101

  ALNS         nv= 16  cost=  1733.6  BKS TD +2.2% NV +2  (24.2s)
  DQN          nv= 38  cost=  4021.7  BKS TD +137.0% NV +24  (7.2s)
  RL-ALNS      nv= 15  cost=  1720.3  BKS TD +1.4% NV +1  (40.4s)

✓ Smoke test passed


In [16]:
# ── 16. Phase 1 — Train ALNS + DQN + RL-ALNS on RC1+RC2 ─────────────────────
# Runtime: n_runs=5 → ~5 hrs T4. Set n_runs=1 for quick test.

RESULT_PATH = os.path.join(OUTPUT_DIR, 'benchmark.csv')

df = run_benchmark(
    instances  = RC1 + RC2,
    algorithms = ['ALNS', 'DQN', 'RL-ALNS'],
    cfg        = CFG,
    result_path= RESULT_PATH,
)
print_paper_table(df)

  Total: 48 combos × 5 runs

  [RC101] ALNS
    run 1/5: nv=15 cost=1739.5 (76.4s)
    run 2/5: nv=15 cost=1641.3 (80.6s)
    run 3/5: nv=16 cost=1682.8 (80.8s)
    run 4/5: nv=16 cost=1674.5 (94.3s)
    run 5/5: nv=15 cost=1721.2 (85.5s)
  → nv=15.4±0.5  td=1691.9±34.8  gap=-0.3%

  [RC101] DQN
    run 1/5: nv=42 cost=4156.5 (1.4s)
    run 2/5: nv=42 cost=4137.2 (1.4s)
    run 3/5: nv=47 cost=4345.0 (1.6s)
    run 4/5: nv=39 cost=4025.2 (1.4s)
    run 5/5: nv=51 cost=4559.7 (1.6s)
  → nv=44.2±4.3  td=4244.7±188.1  gap=+150.1%

  [RC101] RL-ALNS
    run 1/5: nv=15 cost=1664.6 (119.6s)
    run 2/5: nv=15 cost=1693.8 (146.0s)
    run 3/5: nv=15 cost=1716.3 (155.9s)
    run 4/5: nv=15 cost=1715.3 (127.7s)
    run 5/5: nv=15 cost=1707.5 (116.7s)
  → nv=15.0±0.0  td=1699.5±19.2  gap=+0.2%

  [RC102] ALNS
    run 1/5: nv=14 cost=1537.2 (74.6s)
    run 2/5: nv=14 cost=1517.7 (71.9s)
    run 3/5: nv=14 cost=1515.9 (76.6s)
    run 4/5: nv=13 cost=1523.8 (89.3s)
    run 5/5: nv=14 cost=1517.0 (8

In [17]:
# ── 17. Phase 2 — Transfer Learning: Train on RC1, Test on RC2 ───────────────
#
# Train RL-ALNS on RC1 (8 instances), average weights,
# then apply zero-shot to RC2 without any additional training.

# Step 1: Train (or load cached)
transfer_weights = load_transfer_model(CFG)
if transfer_weights is None:
    transfer_weights = train_transfer_model(RC1, CFG, seed=CFG.seed)

# Step 2: Apply to RC2
RESULT_TRANSFER = os.path.join(OUTPUT_DIR, 'benchmark_transfer.csv')
df_tr = run_benchmark(
    instances        = RC2,
    algorithms       = ['RL-ALNS★'],
    cfg              = CFG,
    result_path      = RESULT_TRANSFER,
    transfer_weights = transfer_weights,
)

# Merge with main results for comparison
df_all = pd.concat([df, df_tr], ignore_index=True)
print_paper_table(df_all[df_all['Dataset']=='RC2'])

📚 Training transfer model...
  Training on RC101...
    nv=15, gap=-1.9%
  Training on RC102...
    nv=14, gap=-1.7%
  Training on RC103...
    nv=11, gap=+8.6%
  Training on RC104...
    nv=11, gap=+4.5%
  Training on RC105...
    nv=15, gap=-2.1%
  Training on RC106...
    nv=13, gap=-1.6%
  Training on RC107...
    nv=12, gap=+4.6%
  Training on RC108...
    nv=11, gap=+1.6%

✅ Transfer model saved → /kaggle/working/models/rl_alns_transfer.safetensors
  Total: 8 combos × 5 runs

  [RC201] RL-ALNS★
    run 1/5: nv=5 cost=1386.0 (149.8s)
    run 2/5: nv=4 cost=1459.5 (152.8s)
    run 3/5: nv=5 cost=1340.2 (149.6s)
    run 4/5: nv=4 cost=1522.9 (152.8s)
    run 5/5: nv=5 cost=1363.6 (153.6s)
  → nv=4.6±0.5  td=1414.4±67.4  gap=+0.5%

  [RC202] RL-ALNS★
    run 1/5: nv=4 cost=1278.2 (126.7s)
    run 2/5: nv=4 cost=1241.2 (130.3s)
    run 3/5: nv=4 cost=1181.3 (128.5s)
    run 4/5: nv=4 cost=1196.9 (129.7s)
    run 5/5: nv=4 cost=1255.3 (127.5s)
  → nv=4.0±0.0  td=1230.6±36.2  gap=-9.9%


In [18]:
# ── 18. Full Analysis ─────────────────────────────────────────────────────────
# Load all results
dfs = []
for p in [RESULT_PATH, RESULT_TRANSFER]:
    if os.path.exists(p): dfs.append(pd.read_csv(p))
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if not df_all.empty:
    print('='*60)
    print('FULL RESULTS TABLE')
    print_paper_table(df_all)

    print('\n' + '='*60)
    print('STATISTICAL TESTS')
    print_stats_table(df_all)

    print('\n' + '='*60)
    print('DASHBOARD')
    plot_dashboard(df_all)

    print('\n' + '='*60)
    print('TRANSFER: RL-ALNS★ vs ALNS on RC2')
    plot_transfer_comparison(df_all)
else:
    print('Run cells 16 and 17 first.')

FULL RESULTS TABLE

─────────────────────────────────────────────────────────────────────────────────
DS  Algorithm       NV   ±  vs BKS       TD     ±   Gap% CV_NV CV_TD   OT%   Time
─────────────────────────────────────────────────────────────────────────────────
RC1 ALNS          12.7 0.2    +1.2   1401.6  17.7  +1.6%   1.5   1.2 100.0  82.3s
RC1 DQN           34.5 2.0   +23.1   4069.4 127.3+199.8%   5.6   3.1 100.0   1.4s
RC1 RL-ALNS       12.6 0.1    +1.1   1400.5  17.4  +1.4%   1.1   1.2 100.0 150.7s
─────────────────────────────────────────────────────────────────────────────────
RC2 ALNS           3.5 0.2    +0.3   1144.5  26.8  +2.7%   4.3   2.3 100.0  60.9s
RC2 DQN           18.1 2.5   +14.8   4314.7 126.3+300.4%  14.6   2.9 100.0   1.4s
RC2 RL-ALNS        3.5 0.1    +0.2   1147.0  23.6  +2.8%   2.9   2.0 100.0 128.9s
RC2 RL-ALNS★       4.0 0.4    +0.8   1109.6  35.1  -0.2%   9.9   3.1 100.0 121.2s
──────────────────────────────────────────────────────────────────────────────

In [19]:
# ── 19. Per-Instance Tables (Appendix) ───────────────────────────────────────
if not df_all.empty:
    for ds in ['RC1','RC2']:
        sub = df_all[df_all['Dataset']==ds]
        pivot = sub.pivot_table(
            index='Instance', columns='Algorithm',
            values=['NV_mean','TD_mean','Gap%','NV_cv'],
            aggfunc='mean').round(2)
        print(f'\n── {ds} per-instance detail ──')
        print(pivot.to_string())


── RC1 per-instance detail ──
           Gap%                 NV_cv               NV_mean                TD_mean                  
Algorithm  ALNS     DQN RL-ALNS  ALNS   DQN RL-ALNS    ALNS   DQN RL-ALNS     ALNS      DQN  RL-ALNS
Instance                                                                                            
RC101     -0.30  150.14    0.15  3.18  9.64    0.00    15.4  44.2    15.0  1691.87  4244.72  1699.50
RC102     -2.09  171.87   -1.90  2.90  7.09    2.90    13.8  37.4    13.8  1522.32  4226.90  1525.25
RC103      6.37  227.70    6.63  3.39  5.67    3.39    11.8  34.2    11.8  1341.98  4134.55  1345.35
RC104      5.04  257.84    2.24  0.00  3.19    0.00    10.0  28.0    10.0  1192.75  4063.22  1160.97
RC105     -2.47  153.12   -1.00  2.70  3.79    2.82    14.8  38.8    14.2  1589.19  4124.43  1613.12
RC106     -0.22  172.59    0.41  0.00  4.70    0.00    13.0  34.6    13.0  1421.60  3883.63  1430.54
RC107      4.98  222.86    4.07  0.00  5.52    0.00    12.0 

In [20]:
# ── 20. Route Visualisation ───────────────────────────────────────────────────
# Compare best plans on RC101
inst = RC1[0]
for algo, Solver in [('ALNS', ALNSSolver), ('RL-ALNS', RLALNSSolver)]:
    s = Solver(inst, CFG); plan, _ = s.solve(seed=CFG.seed)
    plot_routes(plan)

## 📋 Changes v2 → v3

### 1. Transfer Learning (new)
- Train RL-ALNS on RC1 → average weights → apply zero-shot to RC2
- `RL-ALNS★` column in results table
- `plot_transfer_comparison()`: scatter showing which instances benefit

### 2. Pre-fill Replay Buffer (new)
- `rla_prefill=300` random episodes fill buffer before search starts
- Eliminates ~600-iteration dead zone where RL had no training signal
- RL now learns from iteration 1 (not iteration 600+)

### 3. Separate iteration budgets (new)
- `alns_iterations=1200` (unchanged)
- `rla_iterations=1800` (+600 to compensate for warm-up cost)
- Fair comparison: both algorithms get ~1200 effective search iterations

### 4. Consistency metric — CV (new)
- CV = std/mean × 100% for both NV and TD
- Lower CV = more predictable results → important for production logistics

### 5. Wilcoxon signed-rank test (new)
- Non-parametric significance test for n=8 instances
- Tests RL-ALNS vs ALNS on Gap% and NV per dataset
- ✅ marks p<0.05 in table

### 6. DQN→ALNS removed
- Identical to ALNS (fallback always triggered) → no contribution
- DQN kept as ablation study (shows why pure RL is insufficient)

### 7. Cleaner table format
- All metrics in one table: NV, TD, Gap%, CV, On-time, Time
- Separator between RC1 and RC2 rows